# 决策树

决策树（decision tree）是一个树结构（可以是二叉树或非二叉树）。  
其每个非叶节点表示一个特征属性上的测试，每个分支代表这个特征属性在某个值域上的输出，而每个叶节点存放一个类别。  
使用决策树进行决策的过程就是从根节点开始，测试待分类项中相应的特征属性，并按照其值选择输出分支，直到到达叶子节点，将叶子节点存放的类别作为决策结果。


下图就是一个由四个特征作为决策条件的决策树：

**这里缺个图**

构建决策树，两个问题：
1. 优先选择哪个特征分裂，越快构建决策树越好。（ID3, C4.5, CART）
2. 分裂特征时，如何选择分裂的标准：一个原则：分裂的子节点，越纯越好。

## ID3算法

对于如何优选选择哪个特征进行分裂，有多种算法，ID3算法是其中之一，它与信息熵有关。

对信息熵的了解，参考知乎网站上Pulsar的回答：https://www.zhihu.com/question/22178202/answer/3385409036

熵在信息论中代表随机变量不确定度的度量。一个离散型随机变量X的熵$H(X)$定义为:
$$
H(X) = - \sum\limits_{x \in X}p(x)logp(x)
$$

其中，x是对于随机变量X的每种结果x  
p(x)表示x发生的概率  
所有p(x)logp(x)求和后取负，表示随机变量X的信息熵，也就是X的不确定量。  
由于p(x)总是小于等于1，因此log(x)总是小于等于0，因此H(X)的值总是大于等于0的。

假设数据集有四个特征A,B,C,D，目标值为E。上面的信息熵公式可以用来计算出目标值E的信息熵，其中p(x)表示E列每个种类出现的概率。

根据A列对E列值进行划分的信息熵：

$$
H(AX) = \sum\limits_{a \in A} p(a)H(X_{a})
$$

其中，p(a)表示A列中每种类别的概率，  
$H(X_{a})$表示在a类别下，E列的信息熵

信息增益即为两者差值：

$$
gain(A) = H(AX) - H(X)
$$

gain(A)为小于等于0的值。因为，新增一列的信息后，信息熵肯定是不变或减小了，不可能增加。

下面以一个根据三个特征判断账号是否真实的数据集为例，手动计算下信息增益。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.DataFrame({"日志密度" : ['s','s','l','m','l','m','m','l','m','s'],
                   "好友密度" : ['s','l','m','m','m','l','s','m','s','s'],
                   '是否使用真实头像' : ['no','yes','yes','yes','yes','no','no','no','no','yes'],
                   "账号是否真实" : ['no','yes','yes','yes','yes','yes','no','yes','yes','no']})
df

In [ ]:
# 计算某列的信息熵
def info_single(df, index):
    # 各分类的概率
    p = df[index].value_counts(normalize=True)
    # 信息熵
    info_sum = 0
    for i,col in enumerate(p):
        info_sum += -1 * p[i] * np.log2(p[i])
    return info_sum

In [ ]:
{i : info_single(df, i) for i in df.columns}

In [ ]:
# 求按index列对账号是否真实进行划分的信息熵和增益。
def info_double(df, index):
    # 当前列分类概率
    p = df[index].value_counts(normalize=True)

    # 信息熵和
    info_sum = 0 
    for t in p.index:
        # 过滤出当前类别当前类别
        sub_df = df[df[index] == t]
        # 计算该分类下yes和no的概率
        prob = sub_df['账号是否真实'].value_counts(normalize=True)
        # print(f'======{index}:{t}=======')
        if 'yes' in prob.index:
            # print(f'真实账号概率:{prob["yes"]}')
            info_sum += p[t] * -1 * prob['yes'] * np.log2(prob['yes'])
        if 'no' in prob.index:
            # print(f'虚假账号概率:{prob["no"]}')
            info_sum += p[t] * -1 * prob['no'] * np.log2(prob['no'])
    return info_single(df, index), info_sum, info_sum - info_single(df, '账号是否真实')

In [ ]:
info_double(df, '日志密度')

In [ ]:
pd.DataFrame(data = {i : info_double(df, i) for i in df.columns}, index=['当前列的信息熵','当前列对账号真实分类的信息熵', '信息熵增益'])

结论：使用好友密度对账号真实分类的信息熵增益最高，因此首先使用好友密度进行分类。

  以上是ID3算法选择分类特征的过程。

ID3的优点：简单，好理解。

缺点：1.大量对数运算，对计算机不友好。2. 容易优先对比较离散的特征进行划分，可能会导致分类错误。

## C4.5算法

C4.5算法是对ID3的改良：让信息熵增益除以特征本身的信息熵。人为降低离散特征的信息增益。

使用信息增益率来选择分类特征。

下面新增一列ID数据，来展示ID3和C4.5在分类特征筛选结果上的区别。

In [ ]:
df2 = df.copy()
df2.insert(0, "ID", range(len(df)))

In [ ]:
df2

In [ ]:
# 可以看到，由于ID列的信息熵值很高，因此会优先按ID列划分，这明显不是我们想要的，每个id的账号真实情况确定，但账号是否真实不应该按id划分。
pd.DataFrame(data = {i : info_double(df2, i) for i in df2.columns}, index=['当前列的信息熵','当前列对账号真实分类的信息熵', '信息熵增益'])

C4.5算法使用信息增益率代替信息增益作为特征选择的条件：

$$
信息增益率 = \frac{信息增益}{特征信息熵}
$$

对取值多的特征做惩罚，让选择更合理。

In [ ]:
# 求按index列对账号是否真实进行划分的信息熵和增益率。
def info_rate_double(df, index):
    # 当前列分类概率
    p = df[index].value_counts(normalize=True)

    # 信息熵和
    info_sum = 0 
    for t in p.index:
        # 过滤出当前类别当前类别
        sub_df = df[df[index] == t]
        # 计算该分类下yes和no的概率
        prob = sub_df['账号是否真实'].value_counts(normalize=True)
        # print(f'======{index}:{t}=======')
        if 'yes' in prob.index:
            # print(f'真实账号概率:{prob["yes"]}')
            info_sum += p[t] * -1 * prob['yes'] * np.log2(prob['yes'])
        if 'no' in prob.index:
            # print(f'虚假账号概率:{prob["no"]}')
            info_sum += p[t] * -1 * prob['no'] * np.log2(prob['no'])
    return info_single(df, index), info_sum, (info_sum - info_single(df, '账号是否真实')) / info_single(df, index)

In [ ]:
# 可以看到C4.5算法会优先选择好友密度进行划分。
pd.DataFrame(data = {i : info_rate_double(df2, i) for i in df2.columns}, index=['当前列的信息熵','当前列对账号真实分类的信息熵', '信息熵增益率'])

## CART算法

CART算法: classification and regression tree 分类和回归树  
ID3和C4.5只能做分类任务，CART既可以做分类，也可以做回归  
CART树是二叉树  
CART算法通过计算基尼不纯度：gain impurity  
思想上跟计算信息增益很像，CART把计算信息熵换成了计算基尼系数。

基尼系数：

$$
gini(X) = 1 - \sum\limits_{x \in X} p(x)^2
$$

其中，gini(X)表示随机变量X的基尼系数    
p(x)表示X的其中一种分类x的概率

使用A列对E列划分的基尼系数：

$$
gini(AX) = \sum\limits_{a \in A} p(a)gini(X_{a})
$$

其中，p(a)表示A列的某个分类a的概率  
$gini(X_a)$表示在A列为a的情况下E列的基尼系数。

使用A列对E列进行划分的基尼不纯度：

$$
giniip(A) = gini(AX) - gini(X)
$$

In [ ]:
# 计算某列的基尼系数
def gini_single(df, index):
    # 各分类的概率
    p = df[index].value_counts(normalize=True)
    # 基尼系数
    gini_sum = 1
    for i,col in enumerate(p):
        gini_sum -= p[i] ** 2
    return gini_sum

In [ ]:
{i : gini_single(df2, i) for i in df2.columns}

In [ ]:
# 求按index列对账号是否真实进行划分的基尼系数和不纯度。
def gini_double(df, index):
    # 当前列分类概率
    p = df[index].value_counts(normalize=True)

    # 基尼系数
    gini_sum = 0 
    for t in p.index:
        # 过滤出当前类别当前类别
        sub_df = df[df[index] == t]
        # 计算该分类下yes和no的概率
        prob = sub_df['账号是否真实'].value_counts(normalize=True)
        # print(f'======{index}:{t}=======')
        gini_sum_tmp = 1
        if 'yes' in prob.index:
            # print(f'真实账号概率:{prob["yes"]}')
            gini_sum_tmp -= prob['yes'] ** 2
        if 'no' in prob.index:
            # print(f'虚假账号概率:{prob["no"]}')
            gini_sum_tmp -= prob['no'] ** 2
        gini_sum += p[t] * gini_sum_tmp
    return gini_single(df, index), gini_sum, gini_sum - gini_single(df, '账号是否真实')

In [ ]:
pd.DataFrame(data = {i : gini_double(df2, i) for i in df2.columns}, index=['当前列的基尼系数','当前列对账号真实分类的基尼系数', '基尼不纯度'])

可以看到，CART算法在除ID列外会优先选择好友密度特征进行划分。

CART算法同样会偏好取值多的特征，但是由于不用对数，以及是二叉树分类，不会一次直接按所有类别分叉，因此没有ID3算法偏好程度重。

在实战中可以通过设置最大树深度等参数来解决这个问题。

最优特征在裂分时，会选择让gini系数最低的特征分裂点进行分裂。比如，特征是1到100范围的数，计算后可能小于等于45是最佳分裂点。

## 三种算法的对比

| 对比项 | ID3 | C4.5 | CART |
| --- | --- | --- | --- |
| 划分准则 | 信息增益 | 信息增益率 | 基尼不纯度 |
| 特征类型 | 仅离散 | 离散+连续 | 离散+连续 |
| 树结构 | 多叉树 | 多叉树 | 二叉树 |
| 任务类型 | 分类 | 分类 | 分类+回归 |

## 决策树的可视化

In [ ]:
# 训练数据
df3 = df.copy()
data = df3.iloc[:, :-1].values
target = df3.iloc[:, -1].values

In [ ]:
# 字符串编码
encoder = LabelEncoder()
for i in range(len(data)):
    data[i] = encoder.fit_transform(data[i])

In [ ]:
# 训练
model = DecisionTreeClassifier(criterion='entropy')
model.fit(data, target)

In [ ]:
# 预测评分
y_ = model.predict(data)
model.score(data, target)

In [ ]:
# 画图
tree.plot_tree(model, feature_names=df3.columns[:-1], filled=True)

In [ ]:
# 使用graphviz进行可视化
# 安装：pip install graphviz
# cmake软件安装

# import graphviz
# dot_dat = tree.export_graphviz(model, feature_names=df3.columns[:-1], filled=True)
# graph = graphviz.Source(dot_dat)
# graph

## 对比决策树，KNN，逻辑斯蒂回归

使用决策树，KNN，逻辑斯蒂回归对鸢尾花数据集进行分类。·

In [ ]:
# 读取数据
iris = load_iris()
data = iris.data
target = iris.target

In [ ]:
# 拆分数据
X_train, X_test, y_train, y_test = train_test_split(data, target)

In [ ]:
# 决策树
DecisionTreeClassifier().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# KNN
KNeighborsClassifier().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 逻辑斯蒂回归
LogisticRegression().fit(X_train, y_train).score(X_test, y_test)

## 决策树的优缺点

优点：  
1. 对异常值不敏感，准确率还不错。  
2. 可以分类也可以回归。

缺点：  
1. 容易过拟合。

## 决策树剪枝

一般使用决策树需要对决策树进行剪枝操作。  
所谓剪枝就是在决策树构建前，设置一些参数：  
1. max_depth：树最大深度
2. min_samples_split：最小分隔样本数  
3. min_samples_leaf：最小叶子节点样本数  
最后两个参数一般是样本量比较大时有效果。

 ## 用决策树回归

In [ ]:
arc = np.linspace(-100, 100, 100)

In [ ]:
x = np.cos(arc)
y = np.sin(arc)

In [ ]:
plt.scatter(x, y)
plt.axis('equal')

In [ ]:
# 加一点噪声
x2 = x.copy()
y2 = y.copy()
x2 += np.random.randn(100) * 0.05
y2 += np.random.randn(100) * 0.05
plt.scatter(x2, y2)
plt.axis('equal')

In [ ]:
model = DecisionTreeRegressor()
model.fit(x2.reshape(-1, 1), y2)

In [ ]:
X_test = np.cos(np.linspace(-100, 100, 50)).reshape(-1, 1)
y_test = np.sin(np.linspace(-100, 100, 50))
y_ = model.predict(X_test)

In [ ]:
model.score(X_test, y_test)

In [ ]:
# 查看预测结果
plt.scatter(X_test, y_)
plt.scatter(X_test, y_test)
plt.axis('equal')

## 决策树的分类边界线

画出决策树的分类边界线，注意观察不同参数的分类边界线的效果。

In [ ]:
# 读取鸢尾花数据集
iris = load_iris()
data = iris.data
target = iris.target
feature_names = iris.feature_names

In [ ]:
df = pd.DataFrame(data, columns=feature_names)
df.plot()

In [ ]:
# 用网格线组成点，填满背景
x = np.linspace(data[:, -2].min(), data[:, -2].max(), 1000)
y = np.linspace(data[:, -1].min(), data[:, -1].max(), 1000)
X, Y = np.meshgrid(x, y)

In [ ]:
XY = np.c_[X.ravel(), Y.ravel()]
XY.shape

In [ ]:
import math

max_depth = 10
cols = 2
rows = math.ceil(max_depth / 2)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows *5))
axes = axes.flatten()
for i in range(max_depth):
    y_ = DecisionTreeClassifier(max_depth=i+1).fit(data[:, -2:], target).predict(XY)
    axes[i].pcolormesh(X, Y, y_.reshape(1000, 1000), shading='auto')
    axes[i].scatter(data[:, -2], data[:, -1], c=target, cmap='rainbow')
    axes[i].set_xlabel(feature_names[-2])
    axes[i].set_ylabel(feature_names[-1])
    axes[i].set_title(f'max_depth:{i+1}')

**可以看到，max_depth每多1，会在某个特征方向上划线分割数据集，树越深，划线越多，分的越精细。  
但从上图也可以看出max_depth=3的结果是比较理想的，深度超过3后，就过拟合了。**

## 作业：隐形眼镜类型

In [ ]:
# 从文件读取数据
df = pd.read_csv('../决策树/lenses.txt', sep='\t')
df

In [ ]:
# 特征列为字符串，进行编码。
encoder = LabelEncoder()
df2 = df.copy()
for i in range(4):
    df2.iloc[:, i] = encoder.fit_transform(df2.iloc[:, i])
df2

In [ ]:
# 拆分测试集和训练集。
data = df2.iloc[:, :-1].values
target = df2.iloc[:, -1].values
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=3)
display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# 训练
model = DecisionTreeClassifier(max_depth=3)
model.fit(X_train, y_train)

In [ ]:
# 预测
y_ = model.predict(X_test)

In [ ]:
# 评分
model.score(X_test, y_test)

In [ ]:
model.score(data, target)

In [ ]:
model.get_depth()

这个数据集数据太少了。。

## 随机森林和集成算法

集成算法：  
1. bagging：套袋法，代表算法：随机森林。  
2. boosting：提升法，代表算法：GBDT，XGBoost
3. stacking：堆叠法，没有代表算法，是一种思想。

随机森林就是在决策树的基础上发展出来的一个集成算法：  
随机森林中随机的含义：  
1. 随机样本：生成每棵树的样本是随机的，要求每棵树的样本数相同。  
2. 随机特征：生成每棵树的特征是随机的，要求每棵树的特征数相同。

随机森林做分类：进行投票。  
随机森林做回归：计算均值。

### 集成算法

构造不同模型的思路：  
1. 超参数不同，也就是max_depth等模型固有的参数不同。
2. 特征不同，随机抽取一部分特征。
3. 样本不同，随机抽取一部分样本。（bagging，套袋法。）
4. 数据权重不同，为数据分配权重。

1、bagging：(套袋法)  
对训练集抽样，并行独立训练。  
随机森林是这一类的代表。

2、Boosting：（提升法）  
利用训练集训练出模型，根据本次模型的预测结果，调整训练集。  
然后利用调整后的训练集训练下一个模型。  
串行，需要第一个模型。  
Adaboost，GBDT，Xgboost都是提升法的代表。

### 随机森林

随机森林里的每棵决策树是样本随机，特征随机。

In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
RandomForestClassifier()

随机森林总结：  
主要步骤：  
1. 随机选择样本。  
2. 随机选择特征。  
3. 构建决策树。  
4. 随机森林投票或平均。

优点：  
1. 表现良好。  
2. 可以处理高维度数据。  
3. 辅助进行特征选择。  
4. 得益于bagging可以并行训练。  


缺点：  
1. 对于噪声过大的数据容易过拟合。

### 极限森林

极限森林中的每一个决策树都采用**原始训练集**  
抽样后，每个节点分裂时都进行特征随机抽样。  
从分裂随机特征中筛选最优分裂条件。

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

## GBDT决策树

GBDT：梯度提升决策树 gradient Boosting DecisionTree

信息熵、真实分布、非真实分布

交叉熵：给定真实分布的情况下，使用非真实分布所指定的策略消除系统不确定性所需要付出的努力大小。

交叉熵越低，越接近信息熵，说明策略越接近最优策略，越接近真实分布。

**这里缺了交叉熵公式截图**

**这里缺了sigmoid函数求导截图**

### 梯度提升分类树

损失函数是交叉熵  
概率计算使用sigmoid  
使用mse作为分裂标准

梯度提升分类树是一个集成算法。

作业：使用决策树和梯度提升分类树分别对鸢尾花数据集进行建模，比较两种算法的评分。

### 梯度提升回归树

作业：对比KNN和KNN Bagging集成算法的效果。

作业：对比决策树和决策树 Bagging集成算法的效果。

## Adaboost提升树

Adaboosting中的Ada是adaptive的意思，所以AdaBoosting表示适应增强算法！

Adaboosting的每棵树是对样本的权重做了修改，后一棵树对前一颗树预测错误的样本权重做了增加。最终所有树按一定权重投票决定结果。

初始每个样本权重都相等，每棵树训练完成后对样本权重做调整，然后用调整的结果训练下一棵树。

根据每棵树的正确率对每棵树给与投票权重，最终每棵树投票决定最终预测结果。

**用来解决二分类问题。**

Adaboost也可以做多分类和回归。

## XGBoost

XGBoost是基于GBDT做的优化，主要是目标函数不一样。

XGBoost不在sklearn包里，需要单独安装导入。  
import xgboost as xgb  
from xgboost import XGBClassifier # 分类

**XGBoost参数使用说明：https://blog.csdn.net/Soft_Po/article/details/120372703**

**ROC-AUC模型评价指标介绍：https://blog.csdn.net/Soft_Po/article/details/120413460**

为什么对于样本不平衡的数据集，AUC比准确率更好。

作业：重做一遍课堂超参数搜索建模案例。

## 天池：天猫复购预测之挑战